# Financial Tweet Sentiment Classification — Final Pipeline
## Group XX — Text Mining 2025/2026, NOVA IMS

Single linear flow: load data → preprocess → featurize → train on **full**
training set → predict on test set → write `pred_XX.csv`.

**Runtime target**: < 20 minutes on a CPU laptop.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")

import nltk
for res in ["stopwords", "wordnet", "punkt", "punkt_tab",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(res, quiet=True)

from src import DATA_DIR, FIGURES_DIR, MODELS_DIR, OUTPUTS_DIR, RANDOM_STATE
from src.preprocessing import pp_minimal, pp_transformer, preprocess_corpus
from src.features import TfidfFeaturizer, TransformerFeaturizer
from src.models import get_model
from src.evaluation import compute_metrics, print_classification_report, plot_confusion_matrix

np.random.seed(RANDOM_STATE)
print("Environment ready.")

Environment ready.


## 1. Load Data

In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

X_train_full = train_df["text"].values
y_train_full = train_df["label"].values

X_test_text = test_df["text"].values
test_ids = test_df["id"].values

print(f"Full training set: {len(X_train_full):,} tweets")
print(f"Test set: {len(X_test_text):,} tweets")
print(f"Label distribution: {pd.Series(y_train_full).value_counts().sort_index().to_dict()}")

Full training set: 9,543 tweets
Test set: 2,388 tweets
Label distribution: {0: 1442, 1: 1923, 2: 6178}


## 2. Preprocess

Using the best preprocessing pipeline identified in the experimentation notebook.
Two paths depending on the best model type:
- **TF-IDF path**: pp_minimal → TfidfVectorizer
- **Transformer path**: pp_transformer → frozen embeddings + LogReg head

In [3]:
# Try to load the best config from experimentation
import joblib

best_config_path = MODELS_DIR / "best_config.joblib"
if best_config_path.exists():
    best_config = joblib.load(best_config_path)
    print(f"Best model from experimentation: {best_config}")
    USE_TRANSFORMER = best_config["best_feature"] in ["DistilBERT", "FinBERT", "Twitter-RoBERTa"]
else:
    print("No best config found — using TF-IDF + LogReg as default")
    USE_TRANSFORMER = False

# Preprocess for both paths (TF-IDF is always fast; transformer needed if USE_TRANSFORMER)
print("\nApplying pp_minimal preprocessing...")
X_train_processed = preprocess_corpus(X_train_full, pipeline=pp_minimal)
X_test_processed = preprocess_corpus(X_test_text, pipeline=pp_minimal)

if USE_TRANSFORMER:
    print("Applying pp_transformer preprocessing...")
    X_train_tf_processed = preprocess_corpus(X_train_full, pipeline=pp_transformer)
    X_test_tf_processed = preprocess_corpus(X_test_text, pipeline=pp_transformer)

print("Preprocessing complete.")

Best model from experimentation: {'best_feature': 'Twitter-RoBERTa', 'best_model': 'XGBoost', 'best_macro_f1': np.float64(0.7939480947530434)}

Applying pp_minimal preprocessing...


Applying pp_transformer preprocessing...


Preprocessing complete.


## 3. Feature Extraction

In [4]:
if USE_TRANSFORMER:
    import gc

    # Determine which transformer checkpoint to use
    feat_name = best_config["best_feature"]
    checkpoint_map = {
        "DistilBERT": "distilbert-base-uncased",
        "FinBERT": "ProsusAI/finbert",
        "Twitter-RoBERTa": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    }
    checkpoint = checkpoint_map[feat_name]
    cache_name = feat_name.lower().replace("-", "_")

    # Use small batch size to avoid OOM on CPU
    featurizer = TransformerFeaturizer(
        checkpoint=checkpoint, max_length=128, batch_size=4,
        cache_name=cache_name
    )

    print(f"Extracting {feat_name} embeddings (cached if available)...")
    X_train_feat = featurizer.transform(X_train_tf_processed, cache_suffix="_full_train")

    # Free model memory before extracting test embeddings
    gc.collect()

    X_test_feat = featurizer.transform(X_test_tf_processed, cache_suffix="_test")

    # Free transformer model after extraction — only need the numpy arrays from here
    featurizer.model = None
    featurizer.tokenizer = None
    gc.collect()

    print(f"Features: train={X_train_feat.shape}, test={X_test_feat.shape}")
else:
    # TF-IDF features
    tfidf = TfidfFeaturizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                             sublinear_tf=True, max_features=20000)
    X_train_feat = tfidf.fit_transform(X_train_processed)
    X_test_feat = tfidf.transform(X_test_processed)
    print(f"TF-IDF features: train={X_train_feat.shape}, test={X_test_feat.shape}")

Extracting Twitter-RoBERTa embeddings (cached if available)...
Loading cached embeddings from /Users/samuel/VSC/MSDAA-Text-Mining/group_XX/models/embeddings_twitter_roberta_full_train.npy
Loading cached embeddings from /Users/samuel/VSC/MSDAA-Text-Mining/group_XX/models/embeddings_twitter_roberta_test.npy
Features: train=(9543, 768), test=(2388, 768)


## 4. Train Best Model on Full Training Set

In [5]:
import joblib

# Use the best classifier identified in experimentation
best_model_name = best_config.get("best_model", "LogReg") if USE_TRANSFORMER else "LogReg"

if best_model_name == "XGBoost":
    clf = get_model("xgboost")
    print(f"Using XGBoost classifier (best from experimentation)")
elif best_model_name == "SVM":
    from sklearn.svm import LinearSVC
    clf = LinearSVC(class_weight="balanced", max_iter=5000, C=1.0, random_state=RANDOM_STATE)
    print(f"Using LinearSVC classifier")
else:
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0,
                              random_state=RANDOM_STATE)
    print(f"Using LogisticRegression classifier")

# Train on FULL training set (not the 80% split used during experimentation)
clf.fit(X_train_feat, y_train_full)
print(f"Model trained on {len(y_train_full):,} examples.")

# Quick sanity: predict on training set to check the model is reasonable
train_preds = clf.predict(X_train_feat)
train_metrics = compute_metrics(y_train_full, train_preds)
print(f"Training set macro-F1: {train_metrics['macro_f1']:.4f} (sanity check)")

Using XGBoost classifier (best from experimentation)


Model trained on 9,543 examples.
Training set macro-F1: 0.9999 (sanity check)


## 5. Generate Predictions

In [6]:
# Predict on test set
test_preds = clf.predict(X_test_feat)

# Create output DataFrame
pred_df = pd.DataFrame({
    "id": test_ids,
    "label": test_preds,
})

# Validation checks
assert len(pred_df) == len(test_df), f"Expected {len(test_df)} rows, got {len(pred_df)}"
assert list(pred_df.columns) == ["id", "label"], f"Wrong columns: {pred_df.columns.tolist()}"
assert pred_df["label"].isin([0, 1, 2]).all(), "Found labels outside {0,1,2}"
assert pred_df.isna().sum().sum() == 0, "Found NaN values"

# Save
output_path = OUTPUTS_DIR / "pred_XX.csv"
pred_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")
print(f"\nPrediction distribution:")
print(pred_df["label"].value_counts().sort_index())
print(f"\nFirst 10 predictions:")
pred_df.head(10)

Predictions saved to /Users/samuel/VSC/MSDAA-Text-Mining/group_XX/outputs/pred_XX.csv

Prediction distribution:
label
0     309
1     392
2    1687
Name: count, dtype: int64

First 10 predictions:


,id,label
0,0,1
1,1,2
2,2,2
3,3,2
4,4,2
5,5,2
6,6,2
7,7,0
8,8,2
9,9,2


In [7]:
# Final verification
print("=" * 50)
print("FINAL VERIFICATION")
print("=" * 50)
verify_df = pd.read_csv(OUTPUTS_DIR / "pred_XX.csv")
print(f"Rows: {len(verify_df)} (expected: {len(test_df)})")
print(f"Columns: {list(verify_df.columns)}")
print(f"Labels in {{0,1,2}}: {verify_df['label'].isin([0,1,2]).all()}")
print(f"No NaNs: {verify_df.isna().sum().sum() == 0}")
print(f"\nLabel distribution:")
print(verify_df["label"].value_counts().sort_index())
print("\n✓ All checks passed. pred_XX.csv is ready for submission.")

FINAL VERIFICATION
Rows: 2388 (expected: 2388)
Columns: ['id', 'label']
Labels in {0,1,2}: True
No NaNs: True

Label distribution:
label
0     309
1     392
2    1687
Name: count, dtype: int64

✓ All checks passed. pred_XX.csv is ready for submission.
